# Colab training — Akkadian → English (ByT5)

Тонкая обёртка: клонирует `ml-dev`, ставит пакет, готовит данные, запускает обучение через CLI.
Чекпоинты пушатся в **приватный репозиторий на HF Hub** — Google Drive не нужен,
после обрыва сессии обучение продолжается с последнего чекпоинта с Hub.

**Перед запуском** добавь в Colab Secrets (ключик слева, у каждого включи Notebook access):
- `HF_TOKEN` — huggingface.co → Settings → Access Tokens (тип **Write**)
- `WANDB_API_KEY` — wandb.ai/authorize
- `KAGGLE_USERNAME`, `KAGGLE_KEY` — kaggle.com → Settings → Create New Token (из kaggle.json)

Runtime → Change runtime type → **T4 GPU**. При обрыве сессии просто перезапусти все ячейки.

In [ ]:
CONFIG = "configs/baseline.yaml"  # <- какой эксперимент запускаем
BRANCH = "ml-dev"

!nvidia-smi -L

In [ ]:
!git clone --branch {BRANCH} https://github.com/ObjoradDdd/ml-hits-3-lab.git /content/repo
%cd /content/repo/ml
!pip install -q -e . sacrebleu

In [ ]:
# секреты -> окружение; имя HF-репозитория выводим из токена и конфига
import os, yaml
from google.colab import userdata
for key in ("HF_TOKEN", "WANDB_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print("no secret:", key)

from huggingface_hub import whoami
HF_USER = whoami(os.environ["HF_TOKEN"])["name"]
RUN_NAME = yaml.safe_load(open(CONFIG))["run_name"]
HUB_ID = f"{HF_USER}/akkadian-{RUN_NAME}"
FINAL = yaml.safe_load(open(CONFIG))["output_dir"] + "/final"
print("чекпоинты ->", HUB_ID)

In [ ]:
# данные соревнования
!pip install -q kaggle
!kaggle competitions download -c deep-past-initiative-machine-translation -p data/
!cd data && unzip -oq deep-past-initiative-machine-translation.zip && rm -f publications.csv  # 580MB, не нужен
!python -m akkadian_nmt.data_prep --data_dir=./data --out_dir=./data/processed

In [ ]:
# обучение: чекпоинты пушатся на Hub, резюм с Hub после обрыва — автоматический
!python -m akkadian_nmt.train --config={CONFIG} \
    --push_to_hub=True --hub_model_id={HUB_ID}

In [ ]:
# оценка на dev: greedy vs beam {1,4,8}
!python -m akkadian_nmt.evaluate beam_sweep --model_dirs={FINAL} --max_samples=200

In [ ]:
# полный метрический набор (BLEU + chrF++ + geo-mean + COMET) на лучшей конфигурации
!pip install -q unbabel-comet
!python -m akkadian_nmt.evaluate run --model_dirs={FINAL} --num_beams=4 --comet=True \
    --out_file=data/dev_predictions.json

In [ ]:
# сабмит на Kaggle
!python model.py predict-file --dataset=data/test.csv --model_dir={FINAL}
from google.colab import files
files.download("data/results.csv")  # -> загрузить на Kaggle вручную
# либо: !kaggle competitions submit -c deep-past-initiative-machine-translation -f data/results.csv -m "run"